In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

zip_path = "/content/drive/MyDrive/CropIQ/CropIQ.zip"
extract_path = "/content/full_dataset"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Dataset extracted.")
train_root = '/content/full_dataset/Fruit 360/Training'
val_root   = '/content/full_dataset/Fruit 360/Validation'
test_root  = '/content/full_dataset/Fruit 360/Test'
print("Sample folders:", os.listdir(train_root)[:10])

In [ ]:
import shutil, re

target_classes = [
    "Apple", "Banana", "Mango", "Orange", "Papaya",
    "Grape", "Tomato", "Potato", "Onion", "Lemon",
    "Cucumber", "Pepper", "Carrot", "Eggplant",
    "Cabbage", "Guava", "Pomegranate", "Pineapple",
    "Strawberry", "Watermelon"
]

manual_map = {
    "apple_crimson_snow_1": "Apple",
    "apple_golden_1": "Apple", "apple_golden_2": "Apple", "apple_golden_3": "Apple",
    "apple_granny_smith_1": "Apple", "apple_pink_lady_1": "Apple",
    "apple_red_1": "Apple", "apple_red_2": "Apple", "apple_red_3": "Apple",
    "apple_red_delicios_1": "Apple", "apple_red_yellow_1": "Apple",
    "apple_braeburn_1": "Apple",
    "cabbage_white_1": "Cabbage",
    "eggplant_long_1": "Eggplant",
}

def get_parent_class(folder_name):
    if folder_name in manual_map:
        return manual_map[folder_name]
    match = re.match(r'^([a-zA-Z]+(?:\s+[a-zA-Z]+)*)', folder_name)
    if match:
        prefix = match.group(1).lower()
        for cls in target_classes:
            if cls.lower() == prefix:
                return cls
    return None

dst_root = '/content/merged_dataset'

for split, src_split in [('Training', train_root), ('Validation', val_root), ('Test', test_root)]:
    for cls in target_classes:
        os.makedirs(os.path.join(dst_root, split, cls), exist_ok=True)
    for folder_name in os.listdir(src_split):
        src_folder = os.path.join(src_split, folder_name)
        if not os.path.isdir(src_folder):
            continue
        parent = get_parent_class(folder_name)
        if parent is None:
            continue
        dst_folder = os.path.join(dst_root, split, parent)
        for fname in os.listdir(src_folder):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif')):
                shutil.copy2(os.path.join(src_folder, fname), dst_folder)
        print(f"{folder_name} \u2192 {parent}")

print("\nMerging done. Final counts:")
for cls in target_classes:
    train_cnt = len(os.listdir(os.path.join(dst_root, 'Training', cls)))
    val_cnt = len(os.listdir(os.path.join(dst_root, 'Validation', cls)))
    print(f"{cls}: train={train_cnt}, val={val_cnt}")

In [ ]:
ADD_BACKGROUND_CLASS = True

if ADD_BACKGROUND_CLASS:
    import urllib.request, random

    bg_url = "https://images.cocodataset.org/zips/unlabeled2017.zip"
    bg_zip = "/content/coco_unlabeled.zip"
    bg_dir = "/content/coco_unlabeled"

    if not os.path.exists(bg_dir):
        urllib.request.urlretrieve(bg_url, bg_zip)
        with zipfile.ZipFile(bg_zip, 'r') as z:
            members = z.namelist()[:600]
            for m in members:
                z.extract(m, bg_dir)
        print("Background images downloaded.")
    else:
        print("Background images already present.")

    all_bg = []
    for root, dirs, files in os.walk(bg_dir):
        for f in files:
            if f.endswith('.jpg'):
                all_bg.append(os.path.join(root, f))
    random.shuffle(all_bg)

    train_bg = os.path.join(dst_root, 'Training', 'Background')
    val_bg   = os.path.join(dst_root, 'Validation', 'Background')
    test_bg  = os.path.join(dst_root, 'Test', 'Background')
    os.makedirs(train_bg, exist_ok=True)
    os.makedirs(val_bg, exist_ok=True)
    os.makedirs(test_bg, exist_ok=True)

    n_train = min(400, len(all_bg))
    n_val   = min(100, len(all_bg) - n_train)
    n_test  = min(50, len(all_bg) - n_train - n_val)

    for f in all_bg[:n_train]:
        shutil.copy2(f, train_bg)
    for f in all_bg[n_train:n_train + n_val]:
        shutil.copy2(f, val_bg)
    for f in all_bg[n_train + n_val:n_train + n_val + n_test]:
        shutil.copy2(f, test_bg)

    print("Background class added.")

In [ ]:
available_classes = [
    "Apple", "Banana", "Cabbage", "Carrot",
    "Cucumber", "Eggplant", "Grape", "Onion",
    "Orange", "Papaya", "Pepper", "Strawberry",
    "Tomato"
]

ADD_BACKGROUND_CLASS = True
final_classes = available_classes + (["Background"] if ADD_BACKGROUND_CLASS else [])
print("Training classes:", final_classes)

# Model 1: MobileNetV2

In [ ]:
import cv2, numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def replace_background(img):
    img_float = img.astype(np.float32)
    white_thresh = 240
    mask = (img_float[:,:,0] > white_thresh) & (img_float[:,:,1] > white_thresh) & (img_float[:,:,2] > white_thresh)
    mask = (~mask).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask_3ch = np.stack([mask]*3, axis=-1).astype(np.float32)
    bg_color = np.random.randint(0, 256, (3,), dtype=np.uint8)
    bg = np.full((img.shape[0], img.shape[1], 3), bg_color, dtype=np.uint8)
    bg_float = bg.astype(np.float32)
    composite = img_float * mask_3ch + bg_float * (1.0 - mask_3ch)
    return composite.astype(np.float32)

train_datagen = ImageDataGenerator(
    preprocessing_function=replace_background,
    rescale=1./255,
    rotation_range=20, zoom_range=0.2,
    brightness_range=[0.8, 1.2], horizontal_flip=True,
    width_shift_range=0.2, height_shift_range=0.2
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(dst_root, 'Training'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=final_classes
)
val_generator = val_datagen.flow_from_directory(
    os.path.join(dst_root, 'Validation'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=final_classes
)

print(f"Training samples: {train_generator.samples}, Validation: {val_generator.samples}")
print("Class indices:", train_generator.class_indices)

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D

num_classes = len(final_classes)

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

model_mobilenet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model_mobilenet.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early1 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
checkpoint1 = ModelCheckpoint('/content/drive/MyDrive/CropIQ/best_mobilenet.keras', save_best_only=True)

history_mobilenet = model_mobilenet.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=[early1, checkpoint1]
)

# Phase 2: fine-tune last 20 layers
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early2 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
checkpoint2 = ModelCheckpoint('/content/drive/MyDrive/CropIQ/best_mobilenet_finetuned.keras', save_best_only=True)

history_mobilenet_fine = model_mobilenet.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early2, checkpoint2]
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history_mobilenet.history['accuracy'], label='train_acc')
plt.plot(history_mobilenet.history['val_accuracy'], label='val_acc')
plt.title('MobileNetV2 Phase 1 Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(history_mobilenet.history['loss'], label='train_loss')
plt.plot(history_mobilenet.history['val_loss'], label='val_loss')
plt.title('MobileNetV2 Phase 1 Loss'); plt.legend()
plt.show()

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history_mobilenet_fine.history['accuracy'], label='train_acc')
plt.plot(history_mobilenet_fine.history['val_accuracy'], label='val_acc')
plt.title('MobileNetV2 Fine-tune Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(history_mobilenet_fine.history['loss'], label='train_loss')
plt.plot(history_mobilenet_fine.history['val_loss'], label='val_loss')
plt.title('MobileNetV2 Fine-tune Loss'); plt.legend()
plt.show()

# Model 2: EfficientNetB0

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D

base_model2 = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model2.trainable = False

model_efficientnet = Sequential([
    base_model2,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(len(final_classes), activation='softmax')
])

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-4, decay_steps=2000, decay_rate=0.9
)
model_efficientnet.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
              loss='categorical_crossentropy', metrics=['accuracy'])
model_efficientnet.summary()

In [ ]:
early3 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
checkpoint3 = ModelCheckpoint('/content/drive/MyDrive/CropIQ/best_efficientnet.keras', save_best_only=True)

history_efficientnet = model_efficientnet.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    callbacks=[early3, checkpoint3]
)

In [ ]:
# Phase 2: unfreeze higher blocks
base_model2.trainable = True
for layer in base_model2.layers:
    if 'block5' in layer.name or 'block6' in layer.name or 'block7' in layer.name or 'top' in layer.name:
        layer.trainable = True
    else:
        layer.trainable = False

fine_lr = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-5, decay_steps=1000, decay_rate=0.9
)
model_efficientnet.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=fine_lr),
              loss='categorical_crossentropy', metrics=['accuracy'])

early4 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
checkpoint4 = ModelCheckpoint('/content/drive/MyDrive/CropIQ/best_efficientnet_finetuned.keras', save_best_only=True)

history_efficientnet_fine = model_efficientnet.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=[early4, checkpoint4]
)

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history_efficientnet.history['accuracy'], label='train_acc')
plt.plot(history_efficientnet.history['val_accuracy'], label='val_acc')
plt.title('EfficientNet Phase 1 Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(history_efficientnet.history['loss'], label='train_loss')
plt.plot(history_efficientnet.history['val_loss'], label='val_loss')
plt.title('EfficientNet Phase 1 Loss'); plt.legend()
plt.show()

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history_efficientnet_fine.history['accuracy'], label='train_acc')
plt.plot(history_efficientnet_fine.history['val_accuracy'], label='val_acc')
plt.title('EfficientNet Fine-tune Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(history_efficientnet_fine.history['loss'], label='train_loss')
plt.plot(history_efficientnet_fine.history['val_loss'], label='val_loss')
plt.title('EfficientNet Fine-tune Loss'); plt.legend()
plt.show()

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    os.path.join(dst_root, 'Test'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=final_classes,
    shuffle=False
)

for name, mod in [('MobileNetV2', model_mobilenet), ('EfficientNetB0', model_efficientnet)]:
    test_generator.reset()
    test_loss, test_acc = mod.evaluate(test_generator, verbose=1)
    print(f"{name} - Test Accuracy: {test_acc:.4f}  |  Loss: {test_loss:.4f}")

In [ ]:
import tensorflow as tf

for name, mod in [('mobilenet', model_mobilenet), ('efficientnet', model_efficientnet)]:
    mod.save(f'/content/drive/MyDrive/CropIQ/final_{name}.keras')

    converter = tf.lite.TFLiteConverter.from_keras_model(mod)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()

    with open(f'/content/drive/MyDrive/CropIQ/{name}_model.tflite', 'wb') as f:
        f.write(tflite_model)

labels = list(train_generator.class_indices.keys())
with open('/content/drive/MyDrive/CropIQ/labels.txt', 'w') as f:
    for lbl in labels:
        f.write(lbl + '\n')

print("Done! Files saved in your Drive (CropIQ folder):")
print("- final_mobilenet.keras / mobilenet_model.tflite")
print("- final_efficientnet.keras / efficientnet_model.tflite")
print("- labels.txt")
print("Label order:", labels)